# 第2回 宿題（発展）: 講座の用語集を LoRA で学習させ、用語集検索と比べる

演習ノートブック（`lora_format_tuning.ipynb`）と同じモデル・同じ LoRA 設定で、FAQ の代わりに **講座の用語集（58 語）** を学習させます。
さらに、第3回でエージェントに持たせた **用語集検索ツール（`app/tools.py` の `search_glossary`）の結果をプロンプトに入れる方法** と同じ評価データで比べ、「知識をモデルに覚えさせる」ことと「必要なときに調べさせる」ことの違いを数字と出力で確かめます。

**比べる 3 つの条件**（モデルは同じ `Qwen/Qwen3.5-0.8B`、生成は貪欲法で固定）

| 条件 | 何を変えるか | 入力 |
| --- | --- | --- |
| `base` | 学習なし | system + 質問だけ（用語集は渡さない） |
| `search` | 学習なし ＋ 用語集検索 | system + `search_glossary` の結果 + 質問 |
| `lora` | 用語集で LoRA 学習 | system + 質問だけ（用語集は渡さない） |
| `lora_search`（任意） | LoRA 学習 ＋ 用語集検索 | `search` と同じ入力 |

**評価データ**（60 問。`scripts/make_glossary_sft.py` が作る）

| 区分 | 件数 | 中身 | 正解 |
| --- | --- | --- | --- |
| `seen` | 10 | 学習で見た言い回しそのまま | 定義文・`needs_review: false` |
| `paraphrase` | 24 | 学習に無い言い回し | 同上 |
| `alias` | 10 | 学習に出していない別名で聞く | 同上 |
| `unknown_far` | 8 | 用語集に無い・講座と関係の薄い語 | 「確認が必要」・`needs_review: true` |
| `unknown_near` | 8 | 用語集に無いが、名前や分野が用語集の語に近い語 | 同上 |

**採点**: 形式は演習と同じ。内容は、`needs_review` が正解と一致し、根拠ありの語では answer が **正しい用語の定義に最も近く**（文字 2-gram の Dice 係数で 58 語の定義と比べる）、かつ類似度が `sim_threshold` 以上のときに ○ にします。

**所要時間の目安（T4）**: 準備 5 分、学習前の評価 2 条件で 10〜15 分、学習 10 分前後、学習後の評価 5〜10 分。

## 0. 環境の確認とライブラリの導入

演習ノートブックと同じです。1 つ目のセルに `Tesla T4` と出ることを確かめてから進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定とデータの取得

**やること**: 設定を `CONFIG` にまとめ、自分の fork の `lec02` ブランチを clone して、用語集・検索ツール・データ作成スクリプトを取り込みます。

- データは Colab の中で `scripts/make_glossary_sft.py` を実行して作り直し、リポジトリにコミット済みのファイルと一致することを確かめます（どこで作っても同じデータになることの確認）。
- 学習データは 201 件で、演習（14 件）よりずっと多いので、エポック数は 20 ではなく **4**（約 104 ステップ）にしています。lr・LoRA の r・alpha は演習と同じです。
- 生成する文が長い（定義文は 50〜210 字）ので、`max_new_tokens` は 256 にしています。

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 4,   # 201 件 × 4 周。1 周 26 ステップで合計約 104 ステップ
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,   # 定義文が長いので演習（96）より増やす
    "gen": {"do_sample": False},
    "sim_threshold": 0.5,    # 定義文との類似度（文字 2-gram の Dice）がこれ以上なら内容 ○
    "repo": "https://github.com/UME0325/lecture-ai-engineering-2026",
    "branch": "lec02",
    "work_dir": "/content/glossary_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/aie")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", CONFIG["branch"], CONFIG["repo"], str(REPO)], check=True)
CONFIG["repo_commit"] = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(REPO))
from app.tools import search_glossary, load_glossary   # 第3回の検索ツールをそのまま使う

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
# データを作り直し、コミット済みのファイルと一致するか確かめる
committed = {p.name: p.read_text(encoding="utf-8") for p in (REPO / "data" / "sft").glob("*")}
r = subprocess.run([sys.executable, str(REPO / "scripts" / "make_glossary_sft.py")], capture_output=True, text=True, cwd=REPO)
print(r.stderr)
regenerated = {p.name: p.read_text(encoding="utf-8") for p in (REPO / "data" / "sft").glob("*")}
assert committed == regenerated, "コミット済みのデータと作り直したデータが一致しません"

def read_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
train_rows = read_jsonl(REPO / "data/sft/glossary_sft_train.jsonl")
valid_rows = read_jsonl(REPO / "data/sft/glossary_sft_valid.jsonl")
EVAL = json.loads((REPO / "data/sft/glossary_eval.json").read_text(encoding="utf-8"))
SYSTEM, UNKNOWN_ANSWER, test = EVAL["system"], EVAL["unknown_answer"], EVAL["questions"]
TERMS = load_glossary()
DEF = {t["id"]: t["definition"] for t in TERMS}
NAME = {t["id"]: t["term"] for t in TERMS}
print(f"train={len(train_rows)} valid={len(valid_rows)} test={len(test)} 用語数={len(TERMS)}")
print("学習例の 1 件:", json.dumps(train_rows[0]["messages"][1:], ensure_ascii=False))

## 2. モデルの読み込みと、3 条件の入力の作り方

**やること**: 演習と同じく量子化なし（32bit）でモデルを読み込みます。入力は条件ごとに次のように変えます。

- `base` / `lora`: `質問: ...` だけ（用語集を見せない＝モデルが覚えているかを問う、closed-book）
- `search` / `lora_search`: `search_glossary(<質問中の語>)` の結果を `用語集の検索結果:` として質問の前に入れる（open-book）

検索語には、質問に埋め込んだ語（`surface`）をそのまま渡します。第3回のエージェントでは検索語をモデル自身が選ぶので、ここでの `search` は **検索語の抜き出しが理想的にできた場合の上限** に当たります。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

def build_messages(ex, use_search: bool):
    user = f"質問: {ex['question']}"
    if use_search:
        user = f"用語集の検索結果:\n{search_glossary(ex['surface'])}\n\n" + user
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, ex, use_search: bool):
    prompt = tok.apply_chat_template(build_messages(ex, use_search), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# 検索結果の例（ヒットする語・しない語・近いが別の語）
for s in ["LoRA", "強化学習", "継続事前学習"]:
    print(s, "→", search_glossary(s)[:120])

## 3. 採点の関数と、学習前の評価（`base` と `search`）

**採点**

- **形式**: 演習と同じ `check_format`（JSON・キーがちょうど 2 つ・型）。
- **内容**: `needs_review` が正解と一致し、根拠ありの語では、answer に最も近い定義（58 語中）が正解の語で、類似度が `sim_threshold` 以上。
  - 定義の 1 文目だけを正しく答えた場合、類似度は 0.3〜0.7 程度になります。そのため、類似度を問わず「正しい語の定義に一番近い」だけを見る緩い基準 `identified_rate` も併せて出します（`inaccurate` は緩い基準では ○）。
- **失敗の型**（内容 × のとき）

| 型 | 意味 |
| --- | --- |
| `format_error` | JSON として読めない |
| `escaped` | 用語集にある語なのに「確認が必要」と逃げた |
| `wrong_term` | 別の用語の定義を答えた（最も近い定義が別の語） |
| `inaccurate` | 正しい語に一番近いが、定義から離れている（言い換え・創作が混ざる） |
| `asserted_borrowed` | 用語集に無い語に、既存の用語の定義を流用して断定した |
| `asserted_made_up` | 用語集に無い語に、それらしい説明を作って断定した |

学習前の 2 条件（約 60 問 × 2）で 10〜15 分かかります。

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", 120)

def check_format(text: str):
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def _bigrams(s: str):
    s = re.sub(r"\s", "", s)
    return {s[i:i+2] for i in range(len(s) - 1)}
DEF_BG = {k: _bigrams(v) for k, v in DEF.items()}

def dice(a: set, b: set) -> float:
    return 2 * len(a & b) / (len(a) + len(b)) if a and b else 0.0

def nearest_definition(answer: str):
    bg = _bigrams(answer)
    scores = {k: dice(bg, v) for k, v in DEF_BG.items()}
    best = max(scores, key=scores.get)
    return best, scores[best], scores

def judge(ex, raw):
    ok_fmt, reason, obj = check_format(raw)
    loose = obj
    if not ok_fmt:   # 形式が壊れていても JSON らしい部分が読めれば内容は別に採点する（演習と同じ）
        m = re.search(r"\{.*\}", raw, re.S)
        try: loose = json.loads(m.group(0)) if m else None
        except Exception: loose = None
    if not isinstance(loose, dict) or not isinstance(loose.get("answer"), str):
        return {"format_ok": ok_fmt, "format_reason": reason, "content_ok": False, "fail_type": "format_error", "identified": False,
                "pred_id": None, "sim_pred": 0.0, "sim_gold": None}
    nr = loose.get("needs_review")
    nr = nr if isinstance(nr, bool) else str(nr).lower() == "true"
    pred, sim_pred, scores = nearest_definition(loose["answer"])
    if ex["gold_id"] is not None:   # 用語集にある語
        sim_gold = scores[ex["gold_id"]]
        if nr: ft = "escaped"
        elif pred != ex["gold_id"]: ft = "wrong_term"
        elif sim_gold < CONFIG["sim_threshold"]: ft = "inaccurate"
        else: ft = "ok"
    else:                            # 用語集に無い語
        sim_gold = None
        if nr: ft = "ok"
        else: ft = "asserted_borrowed" if sim_pred >= 0.3 else "asserted_made_up"
    return {"format_ok": ok_fmt, "format_reason": reason, "content_ok": ft == "ok", "fail_type": ft,
            "identified": ft in ("ok", "inaccurate"),   # 緩い基準: 正しい語の定義に一番近い（文面が短い・言い換えでも可）
            "pred_id": pred, "sim_pred": round(sim_pred, 3), "sim_gold": None if sim_gold is None else round(sim_gold, 3)}

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str, use_search: bool):
    rows, t0 = [], time.time()
    for ex in test:
        raw = generate(model, ex, use_search)
        rows.append({"question": ex["question"], "category": ex["category"], "gold_id": ex["gold_id"], "raw": raw, **judge(ex, raw)})
    n = len(rows)
    known = [r for r in rows if r["gold_id"]]
    summary = {"label": label, "n": n,
               "format_rate": round(sum(r["format_ok"] for r in rows) / n, 3),
               "content_rate": round(sum(r["content_ok"] for r in rows) / n, 3),
               "both_rate": round(sum(r["format_ok"] and r["content_ok"] for r in rows) / n, 3),
               "identified_rate": round(sum(r["identified"] for r in rows) / n, 3),
               "mean_sim_gold": round(sum(r["sim_gold"] or 0 for r in known) / len(known), 3),
               "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

RESULTS = {}
RESULTS["base"] = evaluate(model, "base", use_search=False)
RESULTS["search"] = evaluate(model, "search", use_search=True)

In [ ]:
CATS = ["seen", "paraphrase", "alias", "unknown_far", "unknown_near"]

def by_category(results):
    out = {}
    for label, (_, rows) in results.items():
        df = pd.DataFrame(rows)
        out[label] = {c: round(df[df.category == c].content_ok.mean(), 2) for c in CATS}
        out[label]["format"] = round(df.format_ok.mean(), 2)
    return pd.DataFrame(out).T[["format"] + CATS]

def fail_types(results):
    return pd.DataFrame({label: pd.Series([r["fail_type"] for r in rows]).value_counts() for label, (_, rows) in results.items()}).fillna(0).astype(int).T

display(by_category(RESULTS))
display(fail_types(RESULTS))
display(pd.DataFrame([{"区分": r["category"], "質問": r["question"], "base の出力": r["raw"], "base": r["fail_type"],
                       "search の出力": s["raw"], "search": s["fail_type"]}
                      for r, s in zip(RESULTS["base"][1], RESULTS["search"][1])]))

## 4. LoRA 学習（用語集 201 件 × 4 エポック、T4 で 10 分前後）

演習と同じ `LoraConfig`・`SFTConfig` です。変えたのはデータとエポック数・warmup だけです。学習データは JSONL の各行（`messages` 形式）を `Dataset.from_list` にそのまま渡しています。

**見るところ**: `trainable%`、`Training Loss` の推移、学習時間。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_rows)
valid_ds = Dataset.from_list(valid_rows)

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=26, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()   # 途中で止まったら、0〜3 を実行し直してから trainer.train(resume_from_checkpoint=True)
TRAIN_SECONDS = round(time.time() - t0)
LOSSES = [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | loss の推移: {LOSSES}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後の評価（`lora`）と比較

**やること**: 学習後のモデルで、同じ 60 問を closed-book（用語集を渡さない）で評価し、`base`・`search` と並べます。

**見るところ**

- 区分別の内容正解率: `seen` → `paraphrase` → `alias` の順に下がるか（覚えた文を言い回しや別名に結び付けられるか）
- `unknown_far` / `unknown_near`: 学習で「知らない語には確認を求める」形を学んだのに、近い語で断定しないか（`asserted_borrowed` で何の定義を流用したか）
- `lora` と `search` の差: どの区分で、覚えさせる方法が調べる方法に負ける／勝つか

In [ ]:
RESULTS["lora"] = evaluate(tuned, "lora", use_search=False)
display(by_category(RESULTS))
display(fail_types(RESULTS))
# 学習前（base）は通ったのに学習後に落ちた質問
regressed = [b["question"] for b, a in zip(RESULTS["base"][1], RESULTS["lora"][1]) if b["content_ok"] and b["format_ok"] and not (a["content_ok"] and a["format_ok"])]
print("退行:", regressed)
display(pd.DataFrame([{"区分": l["category"], "質問": l["question"], "正解": NAME.get(l["gold_id"], "（用語集に無い）"),
                       "lora の出力": l["raw"], "lora": l["fail_type"], "近い定義": NAME.get(l["pred_id"], ""), "類似度": l["sim_pred"],
                       "search": s["fail_type"]}
                      for l, s in zip(RESULTS["lora"][1], RESULTS["search"][1])]))

## 6. 任意: 学習後のモデルに検索結果も渡す（`lora_search`）

学習で形式と「確認を求める」振る舞いを身に付けたモデルに、検索結果も渡すとどうなるかを見ます（5 分ほど）。学習は closed-book の入力で行ったので、入力の形が学習時と違う点に注意して読みます。

In [ ]:
RESULTS["lora_search"] = evaluate(tuned, "lora_search", use_search=True)
display(by_category(RESULTS))
display(fail_types(RESULTS))

## 7. 結果の保存

`result.json` と `eval_*.json` を `/content/glossary_lab` に保存します。左のファイル一覧からダウンロードし、`lectures/02/homework/` に置いて PR に含めます。

In [ ]:
import transformers, peft, trl
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "data": {"train": len(train_rows), "valid": len(valid_rows), "test": len(test),
             "test_by_category": {c: sum(q["category"] == c for q in test) for c in CATS}},
    "train": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "losses": LOSSES},
    "summaries": {k: v[0] for k, v in RESULTS.items()},
    "content_rate_by_category": by_category(RESULTS).to_dict(orient="index"),
    "fail_types": fail_types(RESULTS).to_dict(orient="index"),
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "config"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for p in WORK.glob("*.json"): shutil.copy(p, Path(CONFIG["drive_dir"]) / p.name)